# Setup & Your First Agent

Welcome to the Hidden Leaf Village. This notebook stands up the `openai-agents` SDK
and builds the **Leaf Village Assistant** — a clerk agent that answers questions
about ninjas, teams, and missions — then runs it three different ways:
`Runner.run_sync`, `Runner.run`, and `Runner.run_streamed`.


## Build the Leaf Village Assistant


In [1]:
import os  # reads OPENAI_API_KEY and OPENAI_MODEL from the environment

from dotenv import load_dotenv  # loads the .env file into the environment
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)
from agents.models import (
    get_default_model,  # gives the SDK's default OpenAI model name
)

load_dotenv()

# OPENAI_MODEL lets you pick a cheaper or newer model without editing the notebook.
model_name = os.environ.get("OPENAI_MODEL", get_default_model())
print("[model]", model_name)

INSTRUCTIONS = (
    "You are the Leaf Village Assistant, a helpful clerk at the Hidden Leaf Village. "
    "You answer questions about ninjas, teams, missions, and ninja ranks (Academy "
    "Student through Kage-level) clearly and briefly, in the style of the Naruto universe."
)

# Initialize the agent
leaf_assistant = Agent(
    # name: the agent's label in traces, also used to name its handoff tool
    name="Leaf Village Assistant",
    # instructions: the system prompt. Here it is plain text, but a function also works
    instructions=INSTRUCTIONS,
    # model: which model answers. Leave it out to use the SDK default
    model=model_name,
)
print("[agent]", leaf_assistant.name)

[model] gpt-5.6-luna
[agent] Leaf Village Assistant


## `Runner.run_sync` is the one method a notebook can't use

An `Agent` is only configuration — a `Runner` is what actually executes it.
`Runner.run` and `Runner.run_streamed` work normally here. `run_sync` is the single
exception, by design: it refuses to start inside a kernel that already has its own
event loop running, which every Jupyter notebook does.


In [ ]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: shows that run_sync cannot start inside a notebook, then shows that Runner.run does the same call fine
async def demo_run_sync_vs_run() -> None:
    # Sets up one fixed reply, tries run_sync (which fails in a notebook),
    # then makes the same call with Runner.run, which works everywhere.
    # Script the model
    scripted_model = ScriptedModel(
        [
            [
                assistant_message(
                    "Naruto Uzumaki is a Kage-level ninja.",
                ),
            ]
        ]
    )

    # Run the Agent — run_sync, which fails inside a notebook because a loop is already running
    try:
        Runner.run_sync(
            leaf_assistant,
            "Who is Naruto Uzumaki?",
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=scripted_model,
                # workflow_name: the name this run shows up under in tracing
                workflow_name="run-sync-demo",
            ),
        )
    except RuntimeError as error:
        # proves run_sync cannot start inside a notebook, because a loop is already running
        print("[expected]", error)

    # Run the Agent — run, the async version that works everywhere
    result = await Runner.run(
        leaf_assistant,
        "Who is Naruto Uzumaki?",
        # run_config: settings for just this run
        run_config=RunConfig(model=scripted_model, workflow_name="run-demo"),
    )
    # final_output: the answer from the last agent, already turned into text
    # proves Runner.run works with the same call, because it does not fight the notebook's loop
    print("[Runner.run]", result.final_output)


await demo_run_sync_vs_run()
# Outside a notebook: asyncio.run(demo_run_sync_vs_run())

## Watch events arrive with `Runner.run_streamed`


In [ ]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: shows events arriving one by one from run_streamed instead of waiting for the final answer
async def demo_streaming() -> None:
    # run_streamed gives control back right away, so you can show progress
    # while the run is still going, instead of waiting for the end.
    # Script the model
    streamed_model = ScriptedModel(
        [
            [
                assistant_message(
                    "Academy Student is the lowest ninja rank; Kage-level is the highest.",
                ),
            ]
        ]
    )

    # Run the Agent
    streamed_result = Runner.run_streamed(
        leaf_assistant,
        "What do ninja ranks mean?",
        # run_config: settings for just this run
        run_config=RunConfig(model=streamed_model, workflow_name="run-streamed-demo"),
    )

    # stream_events(): gives you each event as soon as the run makes it
    # proves run_streamed sends events one by one, and ends with the same
    # final_output that a normal Runner.run would give
    async for event in streamed_result.stream_events():
        print("[event]", event.type)
    print("[final]", streamed_result.final_output)


await demo_streaming()
# Outside a notebook: asyncio.run(demo_streaming())

## Run it live — optional, needs a real key


In [ ]:
import os  # reads OPENAI_API_KEY from the environment

from dotenv import load_dotenv  # loads the .env file into the environment
from agents import (
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)

load_dotenv()


# Demo: one real API call with no run_config, so it uses the real model and shows up in the SDK's tracing
async def demo_live() -> None:
    # No run_config this time. Nothing replaces the model, so the agent
    # calls the real API, and the SDK records it at platform.openai.com/traces.
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Run the Agent
    live_result = await Runner.run(leaf_assistant, "Who is Naruto Uzumaki?")
    print("[live]", live_result.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

## Summary

- An `Agent` is only configuration — `name`, `instructions`, `model` — and nothing
  happens until a `Runner` executes it.
- `Runner.run` and `Runner.run_streamed` both work inside a notebook, while `run_sync`
  refuses to start inside an event loop that is already running.
